In [1]:
import torch

# Check if GPU is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    
    # Memory info
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total memory: {total:.2f} GB")

CUDA available: True
GPU count: 1
GPU name: NVIDIA A100-SXM4-40GB
CUDA version: 12.6
Total memory: 42.47 GB


In [ ]:
# Convert Mind2Web: Arrow → JSON + Images
from datasets import load_dataset
import json
import os
from tqdm import tqdm

# Paths
arrow_path = "datasets/mind2web"
output_json = "datasets/mind2web/mind2web_train.json"
output_images = "datasets/mind2web/images"

# Create images folder
os.makedirs(output_images, exist_ok=True)

# Load Arrow dataset
print("Loading Arrow dataset...")
dataset = load_dataset(arrow_path)
print(f"Dataset: {dataset}")

# Get split
split_name = list(dataset.keys())[0]
data = dataset[split_name]
print(f"Split: {split_name}, Samples: {len(data)}")

# Convert
train_data = []
for idx, item in enumerate(tqdm(data)):
    try:
        img_filename = f"mind2web_{idx:06d}.png"
        img_path = os.path.join(output_images, img_filename)
        
        # Save image
        if item['image'] is not None:
            item['image'].save(img_path)
        else:
            continue
        
        # JSON entry
        train_data.append({
            "id": item.get('id', f"mind2web_{idx:06d}"),
            "image": f"images/{img_filename}",
            "conversations": item['conversations']
        })
    except Exception as e:
        print(f"Error {idx}: {e}")

# Save JSON
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)

print(f"\nDone! Converted {len(train_data)} samples")
print(f"JSON: {output_json}")
print(f"Images: {output_images}")

In [ ]:
# Verify converted data
import json
import os
from PIL import Image

# Check JSON
json_path = "datasets/mind2web/mind2web_train.json"
with open(json_path) as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")
print(f"\nFirst sample:")
print(json.dumps(data[0], indent=2))

# Check image exists
img_path = os.path.join("datasets/mind2web", data[0]['image'])
img = Image.open(img_path)
print(f"\nImage size: {img.size}")
img  # Display image